# 02 — Read NISAR GCOV into xarray

Read NISAR L2 GCOV products from local, S3, or HTTPS paths into `xarray.DataArray`,
explore the HDF5 group structure, extract polarimetric bands with spatial metadata,
convert to dB, and visualize / export results.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/nice-sar/blob/main/notebooks/02_read_gcov.ipynb)

## 1. Install and Import Dependencies

In [ ]:
# Install nice-sar (with pinned fsspec/s3fs for Colab compatibility)
%pip install -q "fsspec<=2025.3.0" "s3fs<=2025.3.0" git+https://github.com/bullocke/nice-sar.git

In [ ]:
import logging
import os

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from nice_sar.auth.earthdata import login, get_s3_filesystem, get_https_filesystem
from nice_sar.io.hdf5 import open_nisar, get_frequencies, get_polarizations
from nice_sar.io.products import read_gcov, read_gcov_metadata, get_projection_info, read_quad_covariances
from nice_sar.preprocess.calibration import linear_to_db
from nice_sar.io.geotiff import export_geotiff, read_band
from nice_sar.viz.display import percentile_stretch

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s", force=True)
logger = logging.getLogger(__name__)

## 2. Authenticate with NASA Earthdata

`login()` checks environment variables, then `~/.netrc`, then prompts interactively.

In [ ]:
login()

# Detect environment — S3 direct read on AWS us-west-2, HTTPS elsewhere
if os.environ.get("AWS_DEFAULT_REGION") == "us-west-2":
    fs = get_s3_filesystem()
    logger.info("Using S3 direct access (us-west-2)")
else:
    fs = get_https_filesystem()
    logger.info("Using HTTPS streaming (outside us-west-2)")

## 3. Search for a GCOV Granule

Search for a NISAR L2 GCOV product over Salt Lake City, Utah.

In [ ]:
import earthaccess

bbox = (-112.1, 40.5, -111.7, 40.9)  # Salt Lake City, UT
results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_BETA_V1",
    bounding_box=bbox,
    count=5,
)
logger.info("Found %d granules", len(results))

# Get the data URL (S3 or HTTPS) for the first result
links = earthaccess.results.DataGranule.data_links(results[0], access="direct")
if not links:
    links = earthaccess.results.DataGranule.data_links(results[0], access="external")
granule_url = links[0]
logger.info("Granule URL: %s", granule_url)

## 4. Open a NISAR GCOV HDF5 File

`open_nisar()` auto-detects local paths, `s3://` URIs, and `https://` URLs.

In [ ]:
h5 = open_nisar(granule_url, filesystem=fs)
logger.info("Opened: %s", granule_url.split("/")[-1])
h5

## 5. Explore HDF5 Group Structure

Walk the HDF5 tree under `/science/LSAR/GCOV/grids/` to see frequencies,
polarization datasets, shapes, and dtypes.

In [ ]:
def print_tree(group, prefix=""):
    """Recursively print HDF5 group/dataset tree."""
    import h5py
    for key in group:
        item = group[key]
        if isinstance(item, h5py.Group):
            logger.info("%s📁 %s/", prefix, key)
            print_tree(item, prefix + "  ")
        else:
            logger.info("%s📄 %s  shape=%s  dtype=%s", prefix, key, item.shape, item.dtype)

root = h5.get("/science/LSAR/GCOV/grids")
if root is not None:
    print_tree(root)

## 6. Extract Available Frequencies and Polarizations

In [ ]:
frequencies = get_frequencies(h5)
logger.info("Frequencies: %s", frequencies)

for freq in frequencies:
    pols = get_polarizations(h5, freq)
    logger.info("  Frequency %s polarizations: %s", freq, pols)

## 7. Read GCOV Metadata and Projection Info

In [ ]:
# Product metadata
meta = read_gcov_metadata(h5)
for k, v in meta.items():
    logger.info("  %s: %s", k, v)

# Projection / CRS / transform
crs, transform, x_coords, y_coords = get_projection_info(h5, frequency="A")
logger.info("CRS: %s", crs)
logger.info("Transform: %s", transform)
logger.info("Extent — X: [%.1f, %.1f]  Y: [%.1f, %.1f]", x_coords[0], x_coords[-1], y_coords[0], y_coords[-1])
logger.info("Pixel spacing: %.1f m (x) × %.1f m (y)", transform.a, transform.e)

## 8. Read a GCOV Band into xarray DataArray

`read_gcov()` returns an `xarray.DataArray` backed by dask with y/x coordinates
and CRS metadata in attrs. Use `"HH"` for diagonal term `HHHH`, etc.

In [ ]:
# Read HH (HHHH diagonal) from frequency A
da_hh = read_gcov(h5, frequency="A", polarization="HH")
da_hh

In [ ]:
# Read HV (HVHV diagonal) from frequency A
da_hv = read_gcov(h5, frequency="A", polarization="HV")

logger.info("HH shape: %s, dtype: %s", da_hh.shape, da_hh.dtype)
logger.info("HV shape: %s, dtype: %s", da_hv.shape, da_hv.dtype)
logger.info("CRS (from attrs): %s", da_hh.attrs["crs"])
logger.info("Y coords range: %.1f to %.1f", float(da_hh.y[0]), float(da_hh.y[-1]))
logger.info("X coords range: %.1f to %.1f", float(da_hh.x[0]), float(da_hh.x[-1]))

## 9. Convert Backscatter to dB Scale

GCOV diagonal terms are stored as linear power (γ⁰). Convert to decibels: $\sigma_{dB} = 10 \log_{10}(\sigma_{linear})$

In [ ]:
# Sanity check
assert linear_to_db(np.array(1.0)) == 0.0
assert np.isclose(linear_to_db(np.array(0.01)), -20.0)

# Convert — works on dask-backed arrays via .values
hh_db = linear_to_db(da_hh.values)
hv_db = linear_to_db(da_hv.values)

# Quick stats on a center subset (avoids edge NaNs)
cy, cx = hh_db.shape[0] // 2, hh_db.shape[1] // 2
subset = hh_db[cy - 128 : cy + 128, cx - 128 : cx + 128]
valid = subset[np.isfinite(subset)]
logger.info("HH dB stats (center 256×256): min=%.1f, max=%.1f, mean=%.1f", valid.min(), valid.max(), valid.mean())

# dB histogram
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.hist(valid.ravel(), bins=100, color="steelblue", edgecolor="none")
ax.set_xlabel("HH backscatter (dB)")
ax.set_ylabel("Count")
ax.set_title("Distribution of HH dB values (center subset)")
plt.tight_layout()
plt.show()

## 10. Read Quad-Pol Covariance Elements

`read_quad_covariances()` reads all available covariance matrix elements from a
frequency grid. Diagonal terms (HHHH, HVHV, VVVV) are real; off-diagonal terms
(HHHV, HHVV, HVVV) are complex.

In [ ]:
grid_path = f"/science/LSAR/GCOV/grids/frequency{frequencies[0]}"
covs = read_quad_covariances(h5, grid_path)

for name, arr in covs.items():
    logger.info("  %s: shape=%s, dtype=%s", name, arr.shape, arr.dtype)

## 11. Visualize Polarization Bands

Display HH and HV in dB as a 1×2 subplot with contrast-enhanced colormaps.

In [ ]:
# Work on a center subset for fast display
cy, cx = hh_db.shape[0] // 2, hh_db.shape[1] // 2
sz = 512
slc = (slice(cy - sz, cy + sz), slice(cx - sz, cx + sz))

hh_sub = hh_db[slc]
hv_sub = hv_db[slc]

# Percentile stretch for contrast
hh_stretched = percentile_stretch(hh_sub, p_low=2, p_high=98)
hv_stretched = percentile_stretch(hv_sub, p_low=2, p_high=98)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(hh_stretched, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("HH (dB, percentile-stretched)")
axes[0].set_xlabel("Pixel X")
axes[0].set_ylabel("Pixel Y")

axes[1].imshow(hv_stretched, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("HV (dB, percentile-stretched)")
axes[1].set_xlabel("Pixel X")

plt.tight_layout()
plt.show()

## 12. Export a Band to GeoTIFF

Write the HH dB band to a Cloud-Optimized GeoTIFF and verify the round-trip.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    out_path = Path(tmpdir) / "hh_db.tif"

    # Export using the transform + CRS from the product
    export_geotiff(hh_db, out_path, transform=transform, crs=crs, description="HH backscatter (dB)")
    logger.info("Wrote: %s (%.1f MB)", out_path.name, out_path.stat().st_size / 1e6)

    # Read back and verify round-trip
    roundtrip, profile = read_band(out_path)
    logger.info("Round-trip shape: %s, CRS: %s", roundtrip.shape, profile.get("crs"))

    # Check the valid pixels match
    valid_mask = np.isfinite(hh_db) & np.isfinite(roundtrip)
    if valid_mask.any():
        max_diff = np.abs(hh_db[valid_mask] - roundtrip[valid_mask]).max()
        logger.info("Max round-trip difference: %.6f", max_diff)

## 13. Cleanup

Close the HDF5 file handle.

In [ ]:
h5.close()
logger.info("Done — HDF5 file closed.")